![Foundry IQ federated retrieval architecture. Three federated Knowledge Sources on the left — Work IQ (tenant work content: mail, chats, files, meetings), Fabric IQ (an airline ontology hosted in Microsoft Fabric), and Web IQ (the Microsoft Grounding "Speedbird" MCP server exposing web, news, videos, and browse tools) — each feed into a single unified Knowledge Base in the center. The Knowledge Base plans subqueries, retrieves across all three sources in parallel, reranks the candidates, and synthesizes one grounded answer with citations. That Knowledge Base is exposed once as the `knowledge_base_retrieve` MCP tool, which is then consumed on the right by a Foundry Agent (via a RemoteTool project connection using ProjectManagedIdentity) and by GitHub Copilot (via a `.vscode/mcp.json` server entry, plus the Copilot CLI and other MCP hosts).](media/microsoft-iq-in-foundry/01-architecture.png)

**Foundry IQ** lets one *Knowledge Base* (KB) fan a single natural-language
question out across many *Knowledge Sources* (KS), then plan, retrieve, rerank,
and **synthesize a cited answer**. The companion recipe
[Mastering Foundry IQ](mastering-foundry-iq) tours every KS type. This one zooms
in on the three **federated** sources that reach *outside* your search index in
real time:

| Source | What it grounds on |
|---|---|
| **Work IQ** | Your tenant's work content — mail, chats, files, meetings — through Microsoft Graph. |
| **Fabric IQ** | A semantic **ontology** in Microsoft Fabric — here, an airline ontology. |
| **Web IQ** | The live web via the Microsoft Grounding **"Speedbird"** MCP server (`web`, `news`, `videos`, `browse`). |

The arc of the recipe: **3 federated Knowledge Sources → one unified Knowledge
Base → a Foundry Agent → GitHub Copilot** — each consumer talking to the *same*
`knowledge_base_retrieve` MCP tool.

> **Preview status & region.** These federated KS types (Work IQ, Fabric
> ontology, MCP Server) are **preview** on the `2026-05-01-preview` Search API.
> Your Search service must live in a
> [preview region](https://learn.microsoft.com/azure/search/search-region-support)
> — this recipe was authored against **West Central US**. GA timing and region
> coverage change; check
> [What's new](https://learn.microsoft.com/azure/search/whats-new) before
> taking a dependency.

> **Note on running this notebook.** Every section degrades gracefully: any
> source whose credential/entitlement is missing prints a `[skipped]` line and
> the rest of the notebook still runs top-to-bottom. The cells here ship with
> **outputs cleared** — they call live Azure / Foundry / Microsoft Grounding
> endpoints, so run them against your own resources.

## 1 · Prerequisites

| Requirement | Notes |
|---|---|
| **Azure AI Search service** | In a [region that provides agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support) (e.g. **West Central US**). Use *Search Service Contributor* (keyless, recommended) or an admin key. |
| **Microsoft Foundry project** | One chat model deployment (e.g. `gpt-4.1-mini`, `gpt-5-mini`, `gpt-4o`) — used by the KB planner/synthesizer and the Foundry Agent in §8. |
| **Work IQ access (gated)** | Work IQ retrieval is **off by default**. Register the `EnableFoundryIQWithWorkIQ` feature flag, re-register the `Microsoft.Search` provider, and have a tenant admin submit the [Work IQ access request form](https://aka.ms/foundry-iq-work-iq-admin-consent-form). Each querying user needs a **Microsoft 365 Copilot license**, and the search service + Work IQ + users must share one Entra tenant. See [§3](#3-·-work-iq-knowledge-source). |
| **Fabric workspace + ontology** | A Fabric workspace with the [ontology tenant settings](https://learn.microsoft.com/fabric/iq/ontology/overview-tenant-settings) enabled and an [ontology item](https://learn.microsoft.com/fabric/iq/ontology/tutorial-1-create-ontology), in the **same Entra tenant** as the search service. |
| **Web IQ MCP API key** | A key for the Microsoft Grounding "Speedbird" MCP server at `https://api.microsoft.ai/v3/mcp`. **Secret** — never commit it. |

> **One token, two per-user sources.** Both Work IQ and Fabric IQ enforce
> permissions at query time via an **on-behalf-of (OBO)** token — you pass the
> *signed-in user's* token (audience `https://search.azure.com/.default`) on the
> retrieve call. §7 mints it for you. Web IQ uses its own stored `x-apikey`.

### Configure your environment

Set these in your shell, or drop them into a local `.env` next to this notebook
(the repo `.gitignore` already excludes `.env`). **The API key is read from the
environment — it is never written into this notebook.**

```bash
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
SEARCH_API_KEY=<your-search-admin-key>
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_API_KEY=<your-azure-openai-key>
AOAI_GPT_DEPLOYMENT=gpt-4.1-mini            # or gpt-5-mini, gpt-4o, ...

# Secret — required only for §5 (Web IQ). Leave blank to skip that source.
WEB_IQ_MCP_API_KEY=<your-speedbird-mcp-key>

# Optional — unlocks §8 (ground a Foundry Agent). Leave blank to skip.
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<project>
FOUNDRY_PROJECT_RESOURCE_ID=/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<acct>/projects/<project>

# Optional — Fabric ontology IDs (defaults below point at the sample airline ontology).
FABRIC_WORKSPACE_ID=d1836af9-9280-46c0-98d6-6c4736a9943b
FABRIC_ONTOLOGY_ID=337db531-62fe-413c-b6fa-7c211b633401
```

### Install dependencies

This recipe pins the public-preview `azure-search-documents` build that exposes
the `2026-05-01-preview` Knowledge Base / Knowledge Source surface, plus
`azure-ai-projects` for the typed Foundry Agent + MCP tool bindings.

In [ ]:
%%capture
# Foundry IQ rides on the public-preview azure-search-documents SDK, which
# exposes the 2026-05-01-preview Knowledge Base / Knowledge Source surface.
# azure-ai-projects brings the typed Foundry Agent + MCP tool bindings. Both
# ship on PyPI -- no extra package feed required.
import importlib.metadata as _md

try:
    _ok = _md.version("azure-search-documents").startswith("12.1.0b")
except _md.PackageNotFoundError:
    _ok = False

if not _ok:
    %pip install --quiet \
        "azure-search-documents==12.1.0b1" \
        "azure-ai-projects==2.1.0" \
        "azure-identity>=1.19.0" \
        "azure-core>=1.32.0" \
        "python-dotenv>=1.0.1" \
        "httpx>=0.28.1"

## 2 · Configure clients + helpers

One `SearchIndexClient` + one `AzureKeyCredential` are reused throughout. We
reuse the same small helpers as the companion recipe:

- `env(name, required=...)` — env-var lookup (with `.env` support) and a
  friendly error. **Secrets like `WEB_IQ_MCP_API_KEY` are read here — never
  hard-coded.**
- `skip(section, reason)` — prints a `[skipped]` line so a missing
  credential never breaks the run.
- `summarize_ks(name)` — prints the SDK status of a KS right after create, so
  you can confirm it isn't stuck.
- `created_ks` — every KS section appends to it on success; §6 builds the
  single KB from whatever it finds.
- `created_resources` — a dict the cleanup section (§10) walks in dependency
  order.

In [ ]:
import os
from typing import Optional

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import __version__ as search_sdk_version
from azure.search.documents.indexes import SearchIndexClient
from dotenv import load_dotenv

load_dotenv(override=True)


def env(name: str, *, required: bool = True, default: Optional[str] = None) -> Optional[str]:
    value = os.getenv(name, default)
    if required and not value:
        raise RuntimeError(f"Missing required env var: {name}")
    return value or None


def foundry_resource_uri(endpoint: str) -> str:
    # Strip any /openai/... path so we have the bare Foundry resource URI.
    return endpoint.split("/openai/", 1)[0].rstrip("/")


# ---- Required ------------------------------------------------------------
SEARCH_ENDPOINT = env("SEARCH_ENDPOINT")
SEARCH_API_KEY = env("SEARCH_API_KEY")
AOAI_ENDPOINT = foundry_resource_uri(env("AOAI_ENDPOINT"))
AOAI_API_KEY = env("AOAI_API_KEY")

GPT_DEPLOYMENT = env("AOAI_GPT_DEPLOYMENT", required=False, default="gpt-4.1-mini")
GPT_MODEL = env("AOAI_GPT_MODEL", required=False, default=GPT_DEPLOYMENT)

# ---- Resource names ------------------------------------------------------
KS_WORK_IQ = "wfw-ks-work-iq"
KS_FABRIC_IQ = "wfw-ks-fabric-iq"
KS_WEB_IQ = "wfw-ks-web-iq"
KB_NAME = "wfw-federated-kb"

credential = AzureKeyCredential(SEARCH_API_KEY)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

# ---- Trackers consumed by the KB section and cleanup --------------------
created_ks: list[str] = []
created_resources: dict[str, str] = {}

print(f"azure-search-documents : {search_sdk_version}")
print(f"Search service         : {SEARCH_ENDPOINT}")
print(f"Foundry endpoint       : {AOAI_ENDPOINT}")
print(f"Chat deployment        : {GPT_DEPLOYMENT} ({GPT_MODEL})")

In [ ]:
import httpx
from azure.core.exceptions import ResourceNotFoundError


def skip(section: str, reason: str) -> None:
    print(f"[skipped] {section}: {reason}")


def summarize_ks(name: str) -> None:
    """Print the SDK status of a Knowledge Source after create."""
    try:
        status = index_client.get_knowledge_source_status(name)
    except Exception as exc:  # pragma: no cover - best-effort summary
        print(f"  status: <unavailable: {exc.__class__.__name__}>")
        return
    sync = getattr(status, "synchronization_status", None) or "?"
    last_state = getattr(status, "last_synchronization_state", None)
    last = getattr(last_state, "status", None) if last_state else "n/a"
    stats = getattr(status, "statistics", None) or {}
    print(f"  synchronizationStatus={sync}  lastSync={last}  stats={stats}")


# Shared httpx client for the two endpoints the typed SDK doesn't yet expose:
# the ARM project-connection PUT in S8, and best-effort cleanup in S10.
http = httpx.Client(timeout=httpx.Timeout(180.0, connect=30.0))

## 3 · Work IQ Knowledge Source

| | |
|---|---|
| **`kind`** | `workIQ` |
| **Grounds on** | Your organization's Microsoft 365 content (mail, chats, files, meetings) via Work IQ. |
| **Pipeline** | Federated — queries Work IQ live at retrieve time; nothing is indexed. |
| **Auth model** | **Per-user OBO.** Each retrieve passes the caller's token via `x-ms-query-source-authorization` (audience `https://search.azure.com/.default`); the engine exchanges it for a Work IQ token and enforces that user's M365 permissions. |

The typed model is intentionally tiny — `workIQ` takes **only** a `name` and
`description` (no parameters object). Creation still requires the gated access
described in §1, so we wrap it in `try/except` and `skip()` cleanly if your
tenant isn't enabled yet.

> **Gated preview.** Work IQ retrieval is off until you register the
> `EnableFoundryIQWithWorkIQ` feature flag, re-register `Microsoft.Search`, and a
> tenant admin completes the [access request form](https://aka.ms/foundry-iq-work-iq-admin-consent-form).
> Each querying user needs a **Microsoft 365 Copilot license**.

> **Latency.** Work IQ can take **40–60 seconds or more** to respond. Keep
> `max_runtime_in_seconds` at **120+** on the retrieve call (we use 180 in §7).

[Create a Work IQ knowledge source](https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-work-iq)

In [ ]:
from azure.search.documents.indexes.models import WorkIQKnowledgeSource

# Work IQ takes no parameters object -- just name + description (kind="workIQ").
ks = WorkIQKnowledgeSource(
    name=KS_WORK_IQ,
    description="Work IQ KS -- tenant mail, chats, files, and meetings via Microsoft Graph.",
)
try:
    index_client.create_or_update_knowledge_source(ks)
    created_ks.append(KS_WORK_IQ)
    print(f"Created {KS_WORK_IQ}")
    summarize_ks(KS_WORK_IQ)
except Exception as exc:
    skip(KS_WORK_IQ, f"create failed (tenant may not be entitled for Work IQ): {exc}")

## 4 · Fabric IQ Ontology Knowledge Source

| | |
|---|---|
| **`kind`** | `fabricOntology` |
| **Grounds on** | A semantic **ontology** in Microsoft Fabric — here, an **airline ontology** (flights, routes, aircraft, crews, and their relationships). Agents answer in business terms instead of reasoning over raw tables. |
| **Pipeline** | Federated — queries the ontology graph in Fabric live. |
| **Auth model** | **Two layers.** *Creation:* the search service's **managed identity** needs access to the Fabric workspace. *Retrieval:* a per-user `query_source_authorization` OBO token (audience `https://search.azure.com/.default`), which the engine exchanges for a Fabric-scoped token (see §7). |

A Fabric ontology is addressed by a **workspace id** + **ontology id** (both
visible in the Fabric portal URL of the ontology item). The defaults below point
at the sample airline ontology; override via `FABRIC_WORKSPACE_ID` /
`FABRIC_ONTOLOGY_ID`.

> **Reasoning-effort constraint.** Fabric Ontology sources **don't support the
> `minimal`** retrieval reasoning effort — use **`low`** or **`medium`**. Our KB
> in §6 uses `low`.

> **Response shape.** Fabric references carry `sourceData.fabricAnswer` (a
> natural-language answer) and `sourceData.fabricRawData` (the grounding data as
> CSV). Set `include_reference_source_data=True` (we do, in §7) to receive them.

> **Managed-identity gotcha.** If creation or retrieval returns a `403`/access
> error, grant the search service identity **Viewer** (or higher) on the Fabric
> workspace, then re-run.

[Create a Fabric Ontology knowledge source](https://learn.microsoft.com/en-us/azure/search/agentic-knowledge-source-how-to-fabric-ontology)

In [ ]:
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
)

# IDs come from the Fabric ontology item URL; defaults target the airline ontology.
FABRIC_WORKSPACE_ID = env(
    "FABRIC_WORKSPACE_ID", required=False, default="d1836af9-9280-46c0-98d6-6c4736a9943b"
)
FABRIC_ONTOLOGY_ID = env(
    "FABRIC_ONTOLOGY_ID", required=False, default="337db531-62fe-413c-b6fa-7c211b633401"
)

ks = FabricOntologyKnowledgeSource(
    name=KS_FABRIC_IQ,
    description="Fabric IQ KS -- airline ontology (flights, routes, aircraft, crews) in Microsoft Fabric.",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
try:
    index_client.create_or_update_knowledge_source(ks)
    created_ks.append(KS_FABRIC_IQ)
    print(f"Created {KS_FABRIC_IQ} (workspace={FABRIC_WORKSPACE_ID}, ontology={FABRIC_ONTOLOGY_ID})")
    summarize_ks(KS_FABRIC_IQ)
except Exception as exc:
    skip(KS_FABRIC_IQ, f"create failed (check Search MI access to the Fabric workspace): {exc}")

## 5 · Web IQ Knowledge Source (MCP Server / "Speedbird")

| | |
|---|---|
| **`kind`** | `mcpServer` |
| **Grounds on** | The live web via the Microsoft Grounding **"Speedbird"** MCP server at `https://api.microsoft.ai/v3/mcp`. |
| **Tools** | `web`, `news`, `videos`, `browse` — each reranked and capped at 4096 output tokens. |
| **Auth model** | A **stored header** (`x-apikey`) carrying your Speedbird MCP key. |
| **Env vars** | `WEB_IQ_MCP_API_KEY` (**secret** — read from the environment, never committed). |

The key is read with `env("WEB_IQ_MCP_API_KEY", required=True)` so it lives only
in your shell/`.env`. If it's missing, the section `skip()`s.

> **Why stored headers and not a Foundry connection?** Speedbird supports both a
> `foundryConnection` auth variant and a `storedHeaders` variant. As of this
> preview, the `foundryConnection` form returns **HTTP 502 "Could not resolve
> the tool manifest"** on KB Retrieve (a known preview bug). The
> `storedHeaders` form (below) sends `x-apikey` directly to the MCP server and
> works reliably — so we use it here.

[Docs](https://learn.microsoft.com/azure/search/agentic-knowledge-source-overview)

In [ ]:
from azure.search.documents.indexes.models import (
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
)

WEB_IQ_MCP_SERVER_URL = "https://api.microsoft.ai/v3/mcp"  # public URL -- fine to commit

# The API key is a SECRET: read it from the environment, never hard-code it.
web_iq_key = env("WEB_IQ_MCP_API_KEY", required=False)

if not web_iq_key:
    skip(KS_WEB_IQ, "set WEB_IQ_MCP_API_KEY (the Speedbird MCP key) to enable Web IQ")
else:
    speedbird_tools = [
        {"name": "web",    "outputParsing": {"kind": "auto"}, "inclusionMode": "reranked", "maxOutputTokens": 4096},
        {"name": "news",   "outputParsing": {"kind": "auto"}, "inclusionMode": "reranked", "maxOutputTokens": 4096},
        {"name": "videos", "outputParsing": {"kind": "auto"}, "inclusionMode": "reranked", "maxOutputTokens": 4096},
        {"name": "browse", "outputParsing": {"kind": "auto"}, "inclusionMode": "reranked", "maxOutputTokens": 4096},
    ]
    ks = McpServerKnowledgeSource(
        name=KS_WEB_IQ,
        description="Web IQ KS -- Microsoft Grounding 'Speedbird' MCP server (web, news, videos, browse).",
        mcp_server_parameters=McpServerKnowledgeSourceParameters(
            server_url=WEB_IQ_MCP_SERVER_URL,
            # storedHeaders auth (x-apikey). NOT foundryConnection -- that variant
            # currently 502s ("Could not resolve the tool manifest") on KB Retrieve.
            authentication=McpServerStoredHeadersAuthentication(
                stored_headers_parameters=McpServerStoredHeadersParameters(
                    headers={"x-apikey": web_iq_key},
                ),
            ),
            tools=speedbird_tools,
        ),
    )
    try:
        index_client.create_or_update_knowledge_source(ks)
        created_ks.append(KS_WEB_IQ)
        print(f"Created {KS_WEB_IQ} ({WEB_IQ_MCP_SERVER_URL})")
        summarize_ks(KS_WEB_IQ)
    except Exception as exc:
        skip(KS_WEB_IQ, f"create failed: {exc}")

## 6 · Build the unified Knowledge Base

One KB consumes whichever of the three federated sources came up. The KB:

| Setting | What it controls |
|---|---|
| `models` | The Azure OpenAI chat model that **plans** subqueries and **synthesizes** the answer. |
| `knowledge_sources` | The federated sources in scope — Work IQ, Fabric IQ, Web IQ. |
| `retrieval_reasoning_effort` | `low` / `medium` — how hard the planner thinks before fanning out. |
| `output_mode` | `extractiveData` (raw chunks) or **`answerSynthesis`** (cited NL answer). |

We pick **`answerSynthesis`** + **`low`** — the production-friendly default.
(`low` also satisfies Fabric IQ, which **rejects `minimal`**.) The cell raises
only if *none* of the three sources were created.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

if not created_ks:
    raise RuntimeError(
        "No Knowledge Sources were created -- cannot build a KB. Enable at least "
        "one of Work IQ / Fabric IQ / Web IQ above."
    )

# Build the KB over whatever federated sources came up this run.
kb_sources = list(created_ks)
print(f"Building KB '{KB_NAME}' over {len(kb_sources)} federated source(s):")
for n in kb_sources:
    print(f"  - {n}")

gpt_params = AzureOpenAIVectorizerParameters(
    resource_url=AOAI_ENDPOINT,
    deployment_name=GPT_DEPLOYMENT,
    api_key=AOAI_API_KEY,
    model_name=GPT_MODEL,
)

kb = KnowledgeBase(
    name=KB_NAME,
    description="Federated KB fanning out across Work IQ, Fabric IQ (airline ontology), and Web IQ.",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=gpt_params)],
    knowledge_sources=[KnowledgeSourceReference(name=n) for n in kb_sources],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions=(
        "Answer using only the retrieved content across all sources. "
        "When a question spans work content, the airline ontology, and the web, "
        "integrate them faithfully and preserve the [ref_id:N] citations."
    ),
)
index_client.create_or_update_knowledge_base(kb)
created_resources["kb"] = KB_NAME
print(f"\nKnowledge Base '{KB_NAME}' is ready.")

## 7 · Hero retrieve — one question across all three sources

This is where federated retrieval earns its keep. The query below bundles
**three concerns** that no single source can answer alone:

1. **Web IQ** — what's the latest *public* news on a topic?
2. **Fabric IQ** — what does our *airline ontology* say about the related
   routes / aircraft?
3. **Work IQ** — what have *we* discussed internally (mail/chats/files)?

The KB **plans** subqueries, **retrieves** each against the right source in
parallel, **reranks**, and **synthesizes** one answer with `[ref_id:N]`
citations. `include_activity=True` exposes every planner step.

> **Per-user sources need a caller token.** Both **Work IQ** and **Fabric IQ**
> are *identity-scoped*: each retrieve must carry a user token via
> `query_source_authorization` (audience `https://search.azure.com`). Without
> it the service responds `Invalid header: 'x-ms-query-source-authorization' is
> null or empty` and those sources return **no references** (if they're the
> *only* sources in the KB, the whole call fails). **Web IQ** authenticates with
> its own stored `x-apikey`, so it grounds with or without the token. The cell
> below mints the signed-in user's token with `DefaultAzureCredential`; the
> token scopes Work IQ to *that user's* M365 content and is used for Fabric
> workspace RBAC. See [agent retrieval auth](https://aka.ms/search-agent-auth).

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    McpServerKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)

retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
    knowledge_base_name=KB_NAME,
)


def user_query_authorization() -> Optional[str]:
    """Mint the signed-in user's token for the per-user federated sources.

    Work IQ and Fabric IQ require a caller token (audience
    https://search.azure.com); Web IQ ignores it. Returns None if a token can't
    be obtained -- in that case those two sources simply return no references.
    """
    try:
        token = DefaultAzureCredential().get_token("https://search.azure.com/.default").token
        print("query_source_authorization : acquired user token")
        return token
    except Exception as exc:  # noqa: BLE001 -- best-effort; Web IQ still works
        print(f"query_source_authorization : unavailable ({exc}); Work IQ + Fabric IQ will return no references")
        return None


query_auth = user_query_authorization()


def ks_params_for(name: str):
    """Per-kind retrieve params for the federated KS wired into the KB."""
    if name == KS_WORK_IQ:
        return WorkIQKnowledgeSourceParams(
            knowledge_source_name=name,
            include_references=True,
            include_reference_source_data=True,
        )
    if name == KS_FABRIC_IQ:
        return FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=name,
            include_references=True,
            include_reference_source_data=True,
        )
    if name == KS_WEB_IQ:
        return McpServerKnowledgeSourceParams(
            knowledge_source_name=name,
            include_references=True,
            include_reference_source_data=True,
        )
    return None


def retrieve(messages: list[dict], max_runtime_seconds: int = 180):
    request = KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role=m["role"],
                content=[KnowledgeBaseMessageTextContent(text=m["content"])],
            )
            for m in messages
        ],
        knowledge_source_params=[p for p in (ks_params_for(n) for n in kb_sources) if p],
        include_activity=True,
        max_runtime_in_seconds=max_runtime_seconds,
    )
    # query_source_authorization auths the per-user sources (Work IQ, Fabric IQ).
    return retrieval_client.retrieve(request, query_source_authorization=query_auth)


def answer_text(result) -> str:
    parts = []
    for message in (result.response or []):
        for content in (message.content or []):
            text = getattr(content, "text", None)
            if text:
                parts.append(text)
    return "\n\n".join(parts)


messages = [
    {
        "role": "user",
        "content": (
            "I'm prepping for a meeting about transatlantic route planning. "
            "First, what's the latest public news on long-haul aircraft and "
            "transatlantic routes? Second, what does our airline ontology say "
            "about the aircraft and routes that serve those destinations? "
            "Third, what have we already discussed internally about this topic?"
        ),
    }
]
result = retrieve(messages)
first_answer = answer_text(result)

print("ANSWER")
print("------")
print(first_answer)
print()
print(f"Activity steps : {len(result.activity or [])}")
print(f"References     : {len(result.references or [])}")


### 7a · Inspect the planner activity trace

The activity trace shows how the planner decomposed the turn and which source
each subquery hit — the clearest window into federated retrieval.

In [ ]:
import json as _json

activity_dicts = [a.as_dict() if hasattr(a, "as_dict") else dict(a) for a in (result.activity or [])]
print(_json.dumps(activity_dicts, indent=2)[:3500])

### 7b · Inspect the first few references (the citations that ground the answer)

Each reference carries the source it came from and its `source_data`, so you can
see Web IQ, Fabric IQ, and Work IQ contributing to the same synthesized answer.
The shapes differ per source: **Web IQ** returns page content, **Fabric IQ**
returns `fabricAnswer` + `fabricRawData` (CSV), and **Work IQ** returns
`extracts[].text` plus a `seeMoreWebUrl` back to Microsoft 365.

In [ ]:
def describe_reference(ref) -> str:
    """Pull a human-readable snippet out of a reference, per source type."""
    rtype = getattr(ref, "type", None)
    src = ref.source_data or {}
    if not isinstance(src, dict):
        return str(src)[:240]
    if rtype == "fabricOntology":                              # Fabric IQ
        bits = []
        if src.get("fabricAnswer"):
            bits.append(str(src["fabricAnswer"]))
        if src.get("fabricRawData"):
            bits.append("data: " + str(src["fabricRawData"])[:160])
        return "  |  ".join(bits) or _json.dumps(src)[:240]
    if rtype == "workIQ":                                      # Work IQ
        texts = [e.get("text") for e in (src.get("extracts") or []) if e.get("text")]
        more = [a.get("seeMoreWebUrl") for a in (src.get("attributions") or []) if a.get("seeMoreWebUrl")]
        out = (" ".join(texts) or src.get("content") or src.get("text") or "")[:200]
        if more:
            out += f"  (see more: {more[0]})"
        return out or _json.dumps(src)[:240]
    # web / mcpServer
    return (src.get("title") or "") + " " + (src.get("content") or src.get("text") or _json.dumps(src))[:200]


for ref in (result.references or [])[:6]:
    rtype = getattr(ref, "type", None)
    score = getattr(ref, "reranker_score", None)
    snippet = describe_reference(ref).strip().replace("\n", " ")
    score_str = f" reranker={score:.2f}" if isinstance(score, (int, float)) else ""
    print(f"[ref_id:{ref.id}] type={rtype!r}{score_str}")
    if snippet:
        print(f"  {snippet[:240]}{'...' if len(snippet) > 240 else ''}")
    print()


## 8 · Ground a Foundry Agent in the Knowledge Base

The Foundry Agent Service hosts the agent loop, the conversation store, and the
MCP runtime. To attach the federated KB:

1. **Create a `RemoteTool` project connection** targeting the KB's MCP URL.
   `authType=ProjectManagedIdentity` lets the project assume its own identity
   when calling Search — no per-user tokens stored on the connection.
2. **Create an agent** with an `mcp` tool that references the connection and
   allow-lists `knowledge_base_retrieve`.
3. **Run a turn** through the Responses API.

> **Auth:** `DefaultAzureCredential` — run `az login` first. Roles: *Cognitive
> Services Contributor* (agent), *Azure AI User* (project endpoint), and write
> access on the project resource group (connection PUT).

> **Skip:** if `FOUNDRY_PROJECT_ENDPOINT` / `FOUNDRY_PROJECT_RESOURCE_ID` are
> blank, this cell prints a skip and the rest of the notebook still runs.

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition

# The KB exposes itself as an MCP server at this URL.
KB_MCP_API_VERSION = "2026-05-01-preview"
MCP_URL = f"{SEARCH_ENDPOINT}/knowledgebases/{KB_NAME}/mcp?api-version={KB_MCP_API_VERSION}"

project_endpoint = env("FOUNDRY_PROJECT_ENDPOINT", required=False)
project_rid = env("FOUNDRY_PROJECT_RESOURCE_ID", required=False)
agent_name = "wfw-cookbook-agent"
connection_name = "wfw-cookbook-connection"

if not (project_endpoint and project_rid):
    skip("Foundry Agent", "set FOUNDRY_PROJECT_ENDPOINT and FOUNDRY_PROJECT_RESOURCE_ID to enable")
else:
    credential_aad = DefaultAzureCredential()
    arm_bearer = get_bearer_token_provider(credential_aad, "https://management.azure.com/.default")

    # 1) Create the RemoteTool project connection pointing at the KB MCP endpoint.
    #    The ARM PUT isn't surfaced by the typed SDK yet, so we use httpx.
    conn_url = (
        f"https://management.azure.com{project_rid}/connections/{connection_name}"
        f"?api-version=2025-10-01-preview"
    )
    conn_body = {
        "name": connection_name,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": MCP_URL,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    }
    conn_resp = http.put(
        conn_url,
        headers={"Authorization": f"Bearer {arm_bearer()}", "Content-Type": "application/json"},
        json=conn_body,
    )
    conn_resp.raise_for_status()
    created_resources["foundry_connection_url"] = conn_url
    print(f"Project connection ready: {connection_name}")

    # 2) Create the Foundry Agent with the typed azure-ai-projects SDK.
    project_client = AIProjectClient(endpoint=project_endpoint, credential=credential_aad)

    mcp_kb_tool = MCPTool(
        server_label="knowledge-base",
        server_url=MCP_URL,
        require_approval="never",
        allowed_tools=["knowledge_base_retrieve"],
        project_connection_id=connection_name,
    )
    agent_def = PromptAgentDefinition(
        model=GPT_DEPLOYMENT,
        instructions=(
            "Always call knowledge_base_retrieve before answering. "
            "The knowledge base federates work content, an airline ontology, and "
            "the live web -- integrate them and preserve [ref_id:N] citations."
        ),
        tools=[mcp_kb_tool],
    )
    agent_version = project_client.agents.create_version(
        agent_name=agent_name,
        definition=agent_def,
    )
    created_resources["foundry_agent"] = agent_name
    print(f"Agent ready: {agent_name} (version {getattr(agent_version, 'version', '?')})")

    # 3) Run a conversation through the Responses API.
    openai_client = project_client.get_openai_client()
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=(
            "Summarize the latest transatlantic route news and tell me which "
            "aircraft in our ontology serve those routes."
        ),
        extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
    )

    print("\nFoundry agent answer")
    print("--------------------")
    print(getattr(response, "output_text", None) or "(no output_text)")

## 9 · Use the Knowledge Base from GitHub Copilot

The very same KB MCP endpoint plugs straight into **GitHub Copilot** in VS Code.
No SDK, no extra service — just point Copilot at the MCP URL.

Create (or edit) **`.vscode/mcp.json`** in your workspace:

```json
{
  "servers": {
    "foundry-iq-kb": {
      "type": "http",
      "url": "https://<your-search>.search.windows.net/knowledgebases/<kb-name>/mcp?api-version=2026-05-01-preview",
      "headers": { "api-key": "${input:search_api_key}" }
    }
  },
  "inputs": [
    {
      "id": "search_api_key",
      "type": "promptString",
      "description": "Azure AI Search query or admin key",
      "password": true
    }
  ]
}
```

**How to fill it in:**

- **`url`** — replace `<your-search>` with your service name and `<kb-name>`
  with the KB name (this recipe uses `wfw-federated-kb`). It's the same string
  as `MCP_URL` printed in §8:
  `{SEARCH_ENDPOINT}/knowledgebases/{KB_NAME}/mcp?api-version=2026-05-01-preview`.
- **`api-key`** — a Search **query or admin key** (Azure portal → your Search
  service → *Keys*). The `${input:search_api_key}` reference makes VS Code
  **prompt** for the key and store it securely, so it's **never hard-coded** in
  the committed file.

**Reload and invoke:**

1. Save `.vscode/mcp.json` and run **MCP: Reload Servers** (or click *Start* on
   the server in the file's CodeLens). Enter the key when prompted.
2. Open **Copilot Chat**, switch to **Agent** mode, and confirm the
   `foundry-iq-kb` server is listed under the tools picker.
3. Ask a question — Copilot calls the `knowledge_base_retrieve` tool and grounds
   its answer (with `[ref_id:N]` citations) on your federated KB.

> The **same URL + header** works for the **GitHub Copilot CLI** and any other
> MCP host (Claude Desktop, the Foundry Agent Service in §8, a custom script) —
> one Knowledge Base, many consumers.

## 10 · Clean up

The KB, the three KS, the Foundry agent, and the project connection are all
chargeable / stateful. We delete them in dependency order (best-effort, each
guarded) so a re-run starts clean. Comment this out to keep the KB around.

In [ ]:
# 1) Delete the Foundry agent + project connection (best-effort).
if created_resources.get("foundry_agent"):
    try:
        project_client.agents.delete(created_resources["foundry_agent"])
        print(f"Deleted Foundry agent: {created_resources['foundry_agent']}")
    except Exception as exc:
        print(f"Foundry agent delete: {exc}")
    try:
        credential_aad = DefaultAzureCredential()
        arm_bearer = get_bearer_token_provider(credential_aad, "https://management.azure.com/.default")
        del_resp = http.delete(
            created_resources["foundry_connection_url"],
            headers={"Authorization": f"Bearer {arm_bearer()}"},
        )
        del_resp.raise_for_status()
        print("Deleted Foundry project connection.")
    except Exception as exc:
        print(f"Foundry connection delete: {exc}")

# 2) Delete the KB.
if created_resources.get("kb"):
    try:
        index_client.delete_knowledge_base(created_resources["kb"])
        print(f"Deleted KB {created_resources['kb']}")
    except ResourceNotFoundError:
        pass
    except Exception as exc:
        print(f"KB delete: {exc}")

# 3) Delete every KS we created (Work IQ, Fabric IQ, Web IQ).
for ks_name in created_ks:
    try:
        index_client.delete_knowledge_source(ks_name)
        print(f"Deleted KS {ks_name}")
    except ResourceNotFoundError:
        pass
    except Exception as exc:
        print(f"KS {ks_name} delete: {exc}")

# 4) Close the shared httpx client.
try:
    http.close()
except Exception:
    pass

## 11 · Next steps

You now have **three federated Knowledge Sources → one Knowledge Base → a
Foundry Agent → GitHub Copilot**, all sharing the same `knowledge_base_retrieve`
MCP tool. From here:

- **Tour every KS type.** The companion recipe
  [Mastering Foundry IQ](mastering-foundry-iq) walks indexed, uploaded, and
  federated sources end-to-end, plus the Microsoft Agent Framework consumer.
- **Wire per-user identity for Work IQ.** Pass the caller's OBO token via
  `x-ms-query-source-authorization` so retrieval is identity-trimmed.
- **Mix in indexed sources.** Add a Search Index / Blob / OneLake KS alongside
  these federated ones for a hybrid KB.

### Reference docs

- [Knowledge Source overview](https://learn.microsoft.com/azure/search/agentic-knowledge-source-overview)
- [Agentic retrieval overview](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Create a Knowledge Base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Connect a KB to a Foundry agent](https://learn.microsoft.com/azure/foundry/agents/how-to/foundry-iq-connect)
- [What's new in Azure AI Search](https://learn.microsoft.com/azure/search/whats-new)